# A Signature of Cosmic-Ray Increase in AD 774-775 from Tree Rings in Japan — Implementation / 구현

**Paper**: Miyake, F., Nagaya, K., Masuda, K., & Nakamura, T. (2012). *Nature*, 486, 240-242.

This notebook implements the key algorithms and concepts from the paper:
이 노트북은 논문의 핵심 알고리즘과 개념을 구현합니다:

1. **Δ¹⁴C data reconstruction & visualization** — Figure 1 재현 / Figure 1 reproduction
2. **Four-box carbon cycle model** — 탄소 순환 모델 시뮬레이션 / Carbon cycle model simulation
3. **Event duration fitting** — Figure 2 재현 (다양한 지속 시간 시나리오) / Figure 2 reproduction (various duration scenarios)
4. **Energy scale estimation** — 에너지 규모 산정 / Energy scale estimation

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from scipy.integrate import solve_ivp

plt.rcParams['figure.figsize'] = (12, 6)
plt.rcParams['font.size'] = 12
plt.rcParams['axes.grid'] = True
plt.rcParams['grid.alpha'] = 0.3

## Part 1: Δ¹⁴C Data Reconstruction & Visualization / Δ¹⁴C 데이터 재구성 및 시각화

논문 Figure 1a의 Δ¹⁴C 데이터를 디지타이즈하여 재구성합니다. Tree A(2년 간격)와 Tree B(1년 간격)의 데이터를 포함합니다.

We reconstruct the digitized Δ¹⁴C data from Figure 1a. Includes Tree A (biennial) and Tree B (yearly) data.

In [ ]:
# Digitized data from Miyake et al. (2012) Figure 1a
# Tree A: biennial measurements, AD 750-820
# Values are approximate Δ¹⁴C (‰) read from the figure

tree_a_years = np.array([
    750, 752, 754, 756, 758, 760, 762, 764, 766, 768,
    770, 772, 774, 776, 778, 780, 782, 784, 786, 788,
    790, 792, 794, 796, 798, 800, 802, 804, 806, 808,
    810, 812, 814, 816, 818, 820
])
tree_a_d14c = np.array([
    -21.0, -22.5, -20.0, -18.5, -19.0, -19.5, -21.0, -22.0, -23.5, -24.0,
    -25.0, -24.5, -25.5, -13.5, -14.0, -15.0, -16.0, -17.0, -17.5, -18.0,
    -19.0, -19.5, -20.0, -20.5, -21.0, -21.5, -22.0, -22.5, -23.5, -24.0,
    -24.5, -25.0, -25.5, -26.0, -26.5, -27.0
])
tree_a_err = np.full_like(tree_a_d14c, 2.6)  # Typical 1 s.d. error

# Tree B: yearly measurements, AD 770-780
tree_b_years = np.array([770, 771, 772, 773, 774, 775, 776, 777, 778, 779, 780])
tree_b_d14c = np.array([
    -25.0, -24.5, -24.0, -25.0, -25.5, -13.0, -13.5, -14.0, -14.5, -15.0, -15.5
])
tree_b_err = np.full_like(tree_b_d14c, 2.6)

print("=== Miyake Event AD 774-775 ===")
print(f"Tree B: Δ¹⁴C at AD 774 = {tree_b_d14c[4]:.1f} ‰")
print(f"Tree B: Δ¹⁴C at AD 775 = {tree_b_d14c[5]:.1f} ‰")
print(f"Increase: {tree_b_d14c[5] - tree_b_d14c[4]:.1f} ‰ in 1 year")
print(f"Typical solar cycle variation: ~2 ‰ over 11 years")
print(f"Ratio: ~{abs(tree_b_d14c[5] - tree_b_d14c[4]) / 2:.0f}× normal solar modulation")

In [ ]:
# Figure 1a reproduction: Δ¹⁴C time series
# 논문 Figure 1a 재현: Δ¹⁴C 시계열

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 6))

# Panel a: Annual/biennial resolution data
ax1.errorbar(tree_a_years, tree_a_d14c, yerr=tree_a_err,
             fmt='v', color='black', markersize=6, capsize=3,
             label='Tree A (biennial)', zorder=3)
ax1.errorbar(tree_b_years, tree_b_d14c, yerr=tree_b_err,
             fmt='o', color='gray', markersize=6, capsize=3,
             markerfacecolor='none', linewidth=1.5,
             label='Tree B (yearly)', zorder=3)

# Highlight the spike region
ax1.axvspan(774, 775, alpha=0.15, color='red', label='AD 774-775 event')
ax1.annotate('~12 ‰ increase\nin 1 year',
             xy=(775, -13), xytext=(785, -8),
             arrowprops=dict(arrowstyle='->', color='red'),
             fontsize=11, color='red', fontweight='bold')

ax1.set_xlabel('Year AD')
ax1.set_ylabel('Δ¹⁴C (‰)')
ax1.set_title('a  Annual Resolution / 연간 해상도', loc='left', fontweight='bold')
ax1.legend(loc='lower left', fontsize=10)
ax1.set_xlim(750, 825)
ax1.set_ylim(-30, 0)

# Panel b: Decadal average vs IntCal comparison
# Compute decadal averages from our data
decades = np.arange(750, 830, 10)
decadal_avg = []
for d in decades:
    mask = (tree_a_years >= d) & (tree_a_years < d + 10)
    if mask.any():
        decadal_avg.append(np.mean(tree_a_d14c[mask]))
    else:
        decadal_avg.append(np.nan)
decadal_avg = np.array(decadal_avg)

# Simulated IntCal98 data (approximate from figure)
intcal_years = np.array([750, 760, 770, 780, 790, 800, 810, 820])
intcal_d14c = np.array([-19.0, -20.0, -22.0, -16.0, -18.5, -20.5, -23.0, -25.5])
intcal_err = np.array([4.0, 3.5, 3.5, 3.5, 3.5, 3.5, 3.5, 4.0])

ax2.errorbar(intcal_years, intcal_d14c, yerr=intcal_err,
             fmt='s', color='gray', markersize=8, capsize=3,
             markerfacecolor='none', linewidth=1.5,
             label='IntCal98', zorder=2)
ax2.errorbar(decades[:len(decadal_avg)], decadal_avg,
             fmt='D', color='black', markersize=7, capsize=3,
             label='Decadal avg (this study)', zorder=3)

ax2.axvspan(770, 790, alpha=0.1, color='red')
ax2.annotate('Spike hidden at\ndecadal resolution!',
             xy=(780, -16), xytext=(795, -8),
             arrowprops=dict(arrowstyle='->', color='red'),
             fontsize=10, color='red', fontstyle='italic')

ax2.set_xlabel('Year AD')
ax2.set_ylabel('Δ¹⁴C (‰)')
ax2.set_title('b  Decadal Average vs IntCal / 10년 평균 비교', loc='left', fontweight='bold')
ax2.legend(loc='lower left', fontsize=10)
ax2.set_xlim(750, 825)
ax2.set_ylim(-30, 0)

fig.suptitle('Figure 1 Reproduction: Miyake et al. (2012)', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

## Part 2: Four-Box Carbon Cycle Model / 4-Box 탄소 순환 모델

논문의 핵심 모델링 도구인 4-box 탄소 순환 모델을 구현합니다. 대기(atmosphere), 생물권(biosphere), 표층 해양(shallow ocean), 심층 해양(deep ocean) 사이의 ¹⁴C 교환을 연립 미분방정식으로 기술합니다.

We implement the four-box carbon cycle model, the paper's core modeling tool. It describes ¹⁴C exchange between atmosphere, biosphere, shallow ocean, and deep ocean via coupled ODEs.

$$\frac{dN_a}{dt} = Q(t) - \lambda N_a - k_{ab}N_a + k_{ba}N_b - k_{as}N_a + k_{sa}N_s$$
$$\frac{dN_b}{dt} = k_{ab}N_a - k_{ba}N_b - \lambda N_b$$
$$\frac{dN_s}{dt} = k_{as}N_a - k_{sa}N_s - k_{sd}N_s + k_{ds}N_d - \lambda N_s$$
$$\frac{dN_d}{dt} = k_{sd}N_s - k_{ds}N_d - \lambda N_d$$

In [ ]:
# Four-box carbon cycle model parameters
# Reference: Büntgen (2018), Güttler et al. (2015), based on Oeschger et al. (1975)

# Carbon reservoir sizes (in GtC = 10^15 g of carbon)
M_a = 50.0    # Atmosphere (pre-industrial ~600 GtC of total C; ~50 GtC ¹⁴C-equivalent)
M_b = 108.0   # Biosphere
M_s = 720.0   # Shallow ocean (mixed layer)
M_d = 36000.0 # Deep ocean

# Transfer coefficients (yr⁻¹)
# These control the exchange rates between boxes
k_ab = 1.0 / 7.7     # Atmosphere → Biosphere (~0.130 yr⁻¹)
k_ba = 1.0 / 16.6    # Biosphere → Atmosphere (~0.060 yr⁻¹)
k_as = 1.0 / 6.3     # Atmosphere → Shallow ocean (~0.159 yr⁻¹)
k_sa = 1.0 / 80.0    # Shallow ocean → Atmosphere (~0.013 yr⁻¹)
k_sd = 1.0 / 28.0    # Shallow ocean → Deep ocean (~0.036 yr⁻¹)
k_ds = 1.0 / 1400.0  # Deep ocean → Shallow ocean (~0.0007 yr⁻¹)

# ¹⁴C decay constant
half_life = 5730.0  # years
lam = np.log(2) / half_life  # ~1.21e-4 yr⁻¹

# Normal ¹⁴C production rate
Q0 = 2.0  # atoms cm⁻² s⁻¹ (global average)

print("=== Four-Box Carbon Cycle Model Parameters ===")
print(f"Reservoir sizes (GtC): Atm={M_a}, Bio={M_b}, Shallow={M_s}, Deep={M_d}")
print(f"¹⁴C half-life: {half_life} years")
print(f"Decay constant λ: {lam:.2e} yr⁻¹")
print(f"\nTransfer coefficients (yr⁻¹):")
print(f"  k_ab = {k_ab:.4f}  (turnover: {1/k_ab:.1f} yr)")
print(f"  k_ba = {k_ba:.4f}  (turnover: {1/k_ba:.1f} yr)")
print(f"  k_as = {k_as:.4f}  (turnover: {1/k_as:.1f} yr)")
print(f"  k_sa = {k_sa:.4f}  (turnover: {1/k_sa:.1f} yr)")
print(f"  k_sd = {k_sd:.4f}  (turnover: {1/k_sd:.1f} yr)")
print(f"  k_ds = {k_ds:.4f}  (turnover: {1/k_ds:.1f} yr)")

In [ ]:
def carbon_cycle_ode(t, y, Q_func):
    """Four-box carbon cycle model ODEs.

    Args:
        t: Time (years).
        y: State vector [N_a, N_b, N_s, N_d] — ¹⁴C content (normalized).
        Q_func: Callable returning production rate at time t.

    Returns:
        Derivatives [dN_a/dt, dN_b/dt, dN_s/dt, dN_d/dt].
    """
    N_a, N_b, N_s, N_d = y
    Q = Q_func(t)

    dN_a = Q - lam * N_a - k_ab * N_a + k_ba * N_b - k_as * N_a + k_sa * N_s
    dN_b = k_ab * N_a - k_ba * N_b - lam * N_b
    dN_s = k_as * N_a - k_sa * N_s - k_sd * N_s + k_ds * N_d - lam * N_s
    dN_d = k_sd * N_s - k_ds * N_d - lam * N_d

    return [dN_a, dN_b, dN_s, dN_d]


def make_event_Q(t_event, duration, extra_production, Q_base=1.0):
    """Create a production rate function with a transient event.

    Args:
        t_event: Event start time (years from t=0).
        duration: Event duration (years).
        extra_production: Total extra ¹⁴C production (normalized units).
        Q_base: Baseline production rate (normalized to 1.0).

    Returns:
        Callable Q(t) returning production rate at time t.
    """
    rate = extra_production / duration if duration > 0 else 0

    def Q_func(t):
        if t_event <= t <= t_event + duration:
            return Q_base + rate
        return Q_base

    return Q_func


def compute_delta14c(N_a_series, N_a_eq):
    """Convert atmospheric ¹⁴C content to Δ¹⁴C (‰).

    Args:
        N_a_series: Atmospheric ¹⁴C time series.
        N_a_eq: Equilibrium atmospheric ¹⁴C content.

    Returns:
        Δ¹⁴C in per mil (‰).
    """
    return (N_a_series / N_a_eq - 1.0) * 1000.0


print("Model functions defined successfully.")

### Steady-State Equilibrium / 정상 상태 평형

먼저 사건 없이 시스템을 충분히 오래 적분하여 정상 상태 평형(steady state)을 구합니다. 이 평형 값이 Δ¹⁴C = 0‰ 기준이 됩니다.

First, we integrate the system for a sufficiently long time without any event to find steady-state equilibrium. This equilibrium value serves as the Δ¹⁴C = 0‰ reference.

In [ ]:
# Find steady-state equilibrium by running the model for 10,000 years
Q_const = lambda t: 1.0  # Normalized baseline production

# Initial guess (arbitrary, will converge to equilibrium)
y0 = [1.0, 1.0, 1.0, 1.0]

# Run to equilibrium
t_span_eq = (0, 10000)
t_eval_eq = np.linspace(0, 10000, 1000)
sol_eq = solve_ivp(carbon_cycle_ode, t_span_eq, y0,
                   args=(Q_const,), t_eval=t_eval_eq,
                   method='RK45', rtol=1e-10, atol=1e-12)

# Extract equilibrium values
N_a_eq = sol_eq.y[0, -1]
N_b_eq = sol_eq.y[1, -1]
N_s_eq = sol_eq.y[2, -1]
N_d_eq = sol_eq.y[3, -1]

print("=== Steady-State Equilibrium (normalized) ===")
print(f"N_a (atmosphere):    {N_a_eq:.6f}")
print(f"N_b (biosphere):     {N_b_eq:.6f}")
print(f"N_s (shallow ocean): {N_s_eq:.6f}")
print(f"N_d (deep ocean):    {N_d_eq:.6f}")
print(f"Total:               {N_a_eq + N_b_eq + N_s_eq + N_d_eq:.6f}")

# Verify equilibrium: derivatives should be ~0
deriv = carbon_cycle_ode(10000, [N_a_eq, N_b_eq, N_s_eq, N_d_eq], Q_const)
print(f"\nEquilibrium check (derivatives ≈ 0):")
print(f"  dN_a/dt = {deriv[0]:.2e}")
print(f"  dN_b/dt = {deriv[1]:.2e}")
print(f"  dN_s/dt = {deriv[2]:.2e}")
print(f"  dN_d/dt = {deriv[3]:.2e}")

## Part 3: Event Duration Fitting (Figure 2 Reproduction) / 사건 지속 시간 적합 (Figure 2 재현)

다양한 지속 시간(0.1, 0.3, 1, 2, 3년)으로 ¹⁴C를 순간 주입하는 시나리오를 시뮬레이션하여 관측 데이터와 비교합니다. 논문 Figure 2를 재현합니다.

We simulate ¹⁴C injection scenarios with various durations (0.1, 0.3, 1, 2, 3 years) and compare with observed data. This reproduces Figure 2 of the paper.

The extra production is tuned so that the resulting Δ¹⁴C spike matches the observed ~12‰ increase.

In [ ]:
# Simulate the Miyake event with different durations
# The event occurs at t=24 (representing AD 774.5 in our model timeline starting at AD 750)
t_event = 24.5  # ~AD 774.5
durations = [0.1, 0.3, 1.0, 2.0, 3.0]
colors = ['#d62728', '#ff7f0e', '#2ca02c', '#1f77b4', '#9467bd']
labels = ['0.1 yr', '0.3 yr', '1 yr', '2 yr', '3 yr']

# Extra production calibrated to produce ~12‰ Δ¹⁴C spike
# This represents ~6×10⁸ atoms cm⁻² equivalent in normalized units
extra_production = 0.068  # Tuned to match observed ~12‰ peak for short durations

# Time span: 70 years total (AD 750 to AD 820)
t_span = (0, 70)
t_eval = np.linspace(0, 70, 2000)

# Initial conditions at equilibrium
y0_eq = [N_a_eq, N_b_eq, N_s_eq, N_d_eq]

# Run simulations for each duration
results = {}
for dur, color, label in zip(durations, colors, labels):
    Q_func = make_event_Q(t_event, dur, extra_production)
    sol = solve_ivp(carbon_cycle_ode, t_span, y0_eq,
                    args=(Q_func,), t_eval=t_eval,
                    method='RK45', rtol=1e-10, atol=1e-12)

    delta14c = compute_delta14c(sol.y[0], N_a_eq)
    results[label] = {
        't': sol.t + 750,  # Convert to AD years
        'delta14c': delta14c,
        'peak': np.max(delta14c)
    }
    print(f"Duration {label}: peak Δ¹⁴C = {results[label]['peak']:.1f} ‰")

print(f"\nObserved increase: ~12 ‰")

In [ ]:
# Figure 2 reproduction: Carbon cycle model simulation results
# 논문 Figure 2 재현: 탄소 순환 모델 시뮬레이션 결과

fig, ax = plt.subplots(figsize=(12, 7))

# Plot model curves for each duration
for dur, color, label in zip(durations, colors, labels):
    r = results[label]
    ax.plot(r['t'], r['delta14c'], color=color, linewidth=2,
            label=f'Δt = {label} (peak: {r["peak"]:.1f} ‰)')

# Overlay observed data (Tree A, shifted to Δ¹⁴C relative to pre-event baseline)
baseline = np.mean(tree_a_d14c[:12])  # Pre-event average
tree_a_relative = tree_a_d14c - baseline
ax.errorbar(tree_a_years, tree_a_relative, yerr=tree_a_err,
            fmt='v', color='black', markersize=7, capsize=3,
            label='Tree A observed', zorder=5)

# Overlay Tree B data
tree_b_relative = tree_b_d14c - baseline
ax.errorbar(tree_b_years, tree_b_relative, yerr=tree_b_err,
            fmt='o', color='black', markersize=7, capsize=3,
            markerfacecolor='none', linewidth=1.5,
            label='Tree B observed', zorder=5)

ax.axhline(y=0, color='gray', linestyle='--', alpha=0.5)
ax.axvline(x=774.5, color='red', linestyle=':', alpha=0.4, label='Event (AD 774.5)')

ax.set_xlabel('Year AD', fontsize=13)
ax.set_ylabel('Δ¹⁴C change (‰)', fontsize=13)
ax.set_title('Figure 2 Reproduction: Four-Box Carbon Cycle Model\n'
             '다양한 사건 지속 시간에 따른 Δ¹⁴C 반응',
             fontsize=14, fontweight='bold')
ax.legend(loc='upper right', fontsize=10, ncol=2)
ax.set_xlim(750, 820)
ax.set_ylim(-5, 18)

plt.tight_layout()
plt.show()

## Part 4: Reservoir Response Analysis / 각 저장소의 반응 분석

사건 발생 후 각 탄소 저장소(대기, 생물권, 표층 해양, 심층 해양)가 어떻게 반응하는지 시각화합니다. 이를 통해 왜 대기 Δ¹⁴C 스파이크가 수십 년에 걸쳐 감쇠하는지 이해할 수 있습니다.

We visualize how each carbon reservoir responds after the event. This explains why the atmospheric Δ¹⁴C spike decays over decades.

In [ ]:
# Run the 0.1-year (instantaneous) scenario and track all reservoirs
Q_inst = make_event_Q(t_event, 0.1, extra_production)
t_span_long = (0, 200)  # 200 years to see full decay
t_eval_long = np.linspace(0, 200, 5000)

sol_long = solve_ivp(carbon_cycle_ode, t_span_long, y0_eq,
                     args=(Q_inst,), t_eval=t_eval_long,
                     method='RK45', rtol=1e-10, atol=1e-12)

# Compute Δ¹⁴C for each reservoir (relative to its equilibrium)
reservoirs = {
    'Atmosphere / 대기': (sol_long.y[0], N_a_eq, '#d62728'),
    'Biosphere / 생물권': (sol_long.y[1], N_b_eq, '#2ca02c'),
    'Shallow Ocean / 표층 해양': (sol_long.y[2], N_s_eq, '#1f77b4'),
    'Deep Ocean / 심층 해양': (sol_long.y[3], N_d_eq, '#ff7f0e'),
}

fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(12, 10), height_ratios=[2, 1])

# Panel 1: Δ¹⁴C response of each reservoir
for name, (data, eq_val, color) in reservoirs.items():
    delta = compute_delta14c(data, eq_val)
    ax1.plot(sol_long.t + 750, delta, color=color, linewidth=2, label=name)

ax1.axvline(x=774.5, color='red', linestyle=':', alpha=0.4)
ax1.axhline(y=0, color='gray', linestyle='--', alpha=0.3)
ax1.set_xlabel('Year AD')
ax1.set_ylabel('Δ¹⁴C change (‰)')
ax1.set_title('Reservoir Response to Instantaneous ¹⁴C Injection (Δt = 0.1 yr)\n'
              '순간 ¹⁴C 주입에 대한 각 저장소의 반응',
              fontsize=13, fontweight='bold')
ax1.legend(fontsize=11)
ax1.set_xlim(750, 950)

# Panel 2: Fractional ¹⁴C distribution over time
total_excess = np.zeros_like(sol_long.t)
excesses = {}
for name, (data, eq_val, color) in reservoirs.items():
    excess = data - eq_val
    excesses[name] = excess
    total_excess += excess

for name, (data, eq_val, color) in reservoirs.items():
    fraction = np.where(total_excess > 1e-15,
                        excesses[name] / total_excess * 100, 0)
    ax2.plot(sol_long.t + 750, fraction, color=color, linewidth=2, label=name)

ax2.axvline(x=774.5, color='red', linestyle=':', alpha=0.4)
ax2.set_xlabel('Year AD')
ax2.set_ylabel('Fraction of excess ¹⁴C (%)')
ax2.set_title('Distribution of Excess ¹⁴C Among Reservoirs / 잉여 ¹⁴C의 저장소별 분포',
              fontsize=13, fontweight='bold')
ax2.legend(fontsize=10)
ax2.set_xlim(750, 950)
ax2.set_ylim(0, 100)

plt.tight_layout()
plt.show()

## Part 5: Energy Scale Estimation / 에너지 규모 산정

Miyake event의 에너지 규모를 다른 태양/우주 사건과 비교합니다. 논문에서 GEANT4로 계산한 양성자 에너지와 ¹⁴C 생성량의 관계를 활용합니다.

We compare the Miyake event's energy scale with other solar/cosmic events. Uses the proton energy vs. ¹⁴C production relationship calculated by GEANT4 in the paper.

In [ ]:
# Energy scale comparison
# All energies in erg (1 erg = 10⁻⁷ J)

events = {
    'Typical solar flare\n일반 태양 플레어': 1e32,
    'X-class flare\nX급 플레어': 1e32,
    'Carrington Event\n캐링턴 사건 (1859)': 5e32,
    'Largest observed SPE\n최대 관측 SPE': 1e33,
    'Miyake Event\n미야케 사건 (AD 775)\n(lower bound)': 1e34,
    'Miyake Event\n미야케 사건 (AD 775)\n(upper bound)': 1e35,
    'Stellar superflare\n항성 superflare': 1e36,
}

fig, ax = plt.subplots(figsize=(14, 6))

event_names = list(events.keys())
event_energies = list(events.values())
colors_bar = ['#4ECDC4', '#45B7D1', '#FFA07A', '#FF6B6B',
              '#d62728', '#8B0000', '#9B59B6']

bars = ax.barh(range(len(events)), [np.log10(e) for e in event_energies],
               color=colors_bar, edgecolor='black', linewidth=0.5, height=0.6)

# Add energy labels
for i, (name, energy) in enumerate(events.items()):
    ax.text(np.log10(energy) + 0.1, i, f'10$^{{{int(np.log10(energy))}}}$ erg',
            va='center', fontsize=11, fontweight='bold')

# Highlight Miyake event range
ax.axvspan(34, 35, alpha=0.1, color='red')

ax.set_yticks(range(len(events)))
ax.set_yticklabels(event_names, fontsize=10)
ax.set_xlabel('log₁₀(Energy) [erg]', fontsize=13)
ax.set_title('Energy Scale Comparison: Solar & Cosmic Events\n'
             '태양 및 우주 사건의 에너지 규모 비교',
             fontsize=14, fontweight='bold')
ax.set_xlim(30, 38)

plt.tight_layout()
plt.show()

# Print numerical comparison
print("=== Energy Scale Summary ===")
print(f"Miyake event energy: 10³⁴ - 10³⁵ erg")
print(f"Carrington event:    ~5×10³² erg")
print(f"Ratio Miyake/Carrington: {1e34/5e32:.0f} - {1e35/5e32:.0f}×")
print(f"\nIf Miyake event occurred today:")
print(f"  - Power grid damage: {1e34/5e32:.0f}-{1e35/5e32:.0f}× worse than Carrington")
print(f"  - GPS/satellite disruption: potentially catastrophic")
print(f"  - Estimated economic damage: trillions of dollars")

## Part 6: Supernova Distance Constraint / 초신성 거리 제약

초신성 기원의 우주선이 관측된 ¹⁴C 증가를 설명하기 위해 필요한 최대 거리를 계산합니다. 우주선 에너지 밀도는 거리의 제곱에 반비례합니다.

We calculate the maximum distance at which a supernova could produce the observed ¹⁴C increase. Cosmic-ray energy density scales inversely with the square of distance.

In [ ]:
# Supernova distance constraint
# A typical core-collapse supernova releases ~10⁵¹ erg total,
# of which ~1% goes into cosmic rays → E_CR ~ 10⁴⁹ erg

E_sn_cr = 1e49  # erg, cosmic-ray energy from supernova
E_earth = 1e35  # erg, energy needed at Earth (upper estimate from paper)

# Earth intercepts a fraction of the isotropic cosmic-ray flux
# Flux at distance d: F = E_CR / (4π d²)
# Earth's cross-section: π R_E² (R_E ~ 6.4×10⁸ cm)
# But we need fluence per cm², so:
# Required fluence: E_earth / (4π d²) = E_needed_per_cm2
# Solving for d: d = sqrt(E_CR / (4π × E_needed_per_cm2))

# From paper: required >30 MeV proton fluence ~3×10⁶ cm⁻²
# Total proton energy corresponding: ~10³⁴-10³⁵ erg at Earth

# Simple estimate: what fraction of SN cosmic-ray energy reaches Earth?
# E_earth / E_sn_cr = (R_earth / d)² / 4
# d = R_earth × sqrt(E_sn_cr / (4 × E_earth))

# More directly from the paper's argument:
# The required cosmic-ray energy density at Earth constrains d
d_max_cm = np.sqrt(E_sn_cr / (4 * np.pi * E_earth))  # cm (simplified)
pc_to_cm = 3.086e18  # 1 parsec in cm
d_max_pc = d_max_cm / pc_to_cm

print("=== Supernova Distance Constraint ===")
print(f"SN cosmic-ray energy: {E_sn_cr:.0e} erg")
print(f"Energy needed at Earth: {E_earth:.0e} erg")
print(f"Maximum SN distance: {d_max_pc:.1f} pc ({d_max_pc * 3.26:.1f} light-years)")
print(f"\nPaper's estimate: ≲ 1 pc (~3.26 light-years)")

# Plot distance vs. required energy
distances_pc = np.logspace(-1, 3, 100)
distances_cm = distances_pc * pc_to_cm
fluence_at_earth = E_sn_cr / (4 * np.pi * distances_cm**2)

fig, ax = plt.subplots(figsize=(10, 6))

ax.loglog(distances_pc, fluence_at_earth, 'b-', linewidth=2,
          label='SN cosmic-ray fluence at Earth')
ax.axhline(y=E_earth, color='red', linestyle='--', linewidth=2,
           label=f'Required for Miyake event (~10³⁵ erg)')
ax.axhline(y=1e34, color='orange', linestyle='--', linewidth=1.5,
           label=f'Required (lower bound, ~10³⁴ erg)')

# Mark key distances
known_snr = {
    'Required\n(~1 pc)': 1.0,
    'Vela Jr.\n(~200 pc)': 200,
    'Vela\n(~290 pc)': 290,
    'Crab Nebula\n(~2000 pc)': 2000,
}
for name, dist in known_snr.items():
    f_at_d = E_sn_cr / (4 * np.pi * (dist * pc_to_cm)**2)
    ax.plot(dist, f_at_d, 'ko', markersize=6, zorder=5)
    ax.annotate(name, (dist, f_at_d),
                textcoords="offset points", xytext=(10, 10),
                fontsize=9, fontstyle='italic')

ax.fill_between(distances_pc, E_earth, fluence_at_earth,
                where=(fluence_at_earth >= E_earth),
                alpha=0.1, color='green', label='SN could explain event')

ax.set_xlabel('Distance to supernova (pc)', fontsize=13)
ax.set_ylabel('Cosmic-ray energy at Earth (erg)', fontsize=13)
ax.set_title('Supernova Distance Constraint for Miyake Event\n'
             '미야케 사건에 대한 초신성 거리 제약',
             fontsize=14, fontweight='bold')
ax.legend(fontsize=10, loc='upper right')
ax.set_xlim(0.1, 3000)
ax.set_ylim(1e30, 1e40)

plt.tight_layout()
plt.show()

print("\nConclusion: No known supernova remnant exists within ~1 pc.")
print("결론: ~1 pc 이내에 알려진 초신성 잔해가 없음 → 초신성 가설 기각.")

## Summary / 요약

| Concept / 개념 | This Paper / 이 논문 | Modern Equivalent / 현대 동등물 |
|---|---|---|
| ¹⁴C 측정 / ¹⁴C measurement | AMS, 2년/1년 해상도 (2 trees) | Single-year resolution, multiple global sites |
| 탄소 순환 모델 / Carbon cycle model | 4-box model (atm-bio-shallow-deep) | 11-box models, coupled climate-carbon models |
| 에너지 추정 / Energy estimation | GEANT4 Monte Carlo (~10³⁴-10³⁵ erg) | Updated GEANT4 + multi-isotope constraints |
| 원인 규명 / Cause identification | Undetermined (superflare or supernova) | Solar proton event confirmed (Mekhaldi 2015) |
| 시간 마커 활용 / Time marker use | First discovery of the spike | Archaeological dating tool (e.g., L'Anse aux Meadows) |

### Key implementation insights / 핵심 구현 시사점

1. **4-box 모델의 효과**: 비교적 간단한 연립 ODE로 Δ¹⁴C 반응의 정성적 특성(피크 높이, 감쇠 타임스케일)을 잘 포착함. 복잡한 모델 없이도 사건 특성화가 가능.
   - **4-box model effectiveness**: Relatively simple coupled ODEs capture qualitative Δ¹⁴C response features (peak height, decay timescale) well. Event characterization is possible without complex models.

2. **시간 해상도의 중요성**: Figure 1 재현에서 같은 데이터라도 10년 평균으로 보면 스파이크가 사라짐. 분석의 시간 해상도가 발견을 결정.
   - **Time resolution matters**: In the Figure 1 reproduction, the spike disappears when the same data is averaged to decadal resolution. Analysis time resolution determines discovery.

3. **에너지 스케일의 극단성**: Miyake event는 Carrington Event보다 20-200배 강력. 현대 인프라 방호 기준을 재설정해야 할 수 있음.
   - **Extreme energy scale**: The Miyake event is 20-200× more powerful than the Carrington Event. Modern infrastructure protection standards may need revision.